- 데이터 정제 후 axencoder-len512에 batch=128.lr=4.8e-4 적용 풀런

In [1]:
import sys
sys.path.insert(0, "src")  # /workspace/src

from patent_train import TrainingRunner, TrainConfig, probe_batches

In [2]:
SEARCH = False   # True=fast-fail 짧은 런(2 epoch로 유도) / False=풀런(아래 epochs 사용)

cfg = TrainConfig(
    backbone="axenc",   # backbones.BACKBONES 키
    loss="focal",
    loss_params={"alpha": 0.25, "gamma": 2},
    seed=153,
    max_len=512,
    eff_batch=128,          # 배치 재현 파라미터
    micro_batch=128,        
    eval_micro_batch=512,
    learning_rate=4.8e-4,   # 확정 레시피 lr
    weight_decay=0.01,
    warmup_ratio=0.1,
    epochs=12,
    early_stop_epochs=2,    # 개선 없이 견디는 에폭 수(eval 횟수 환산은 runner가 처리)
    notebook_name="11_04_Seed153.ipynb",   # wandb code saving
    tag="modernbert-patent-seed153",
    run_name="axenc_len512_seed153",
    repo_final="ingyoun/A.X-patent-seed153",
    out_path="/workspace/output/modernbert-seed153",
    search=SEARCH,
)

print("run_name:", cfg.run_name, "| epochs:", cfg.epochs, "| grad_accum:", cfg.grad_accum)
print(cfg.seed)

run_name: axenc_len512_seed153 | epochs: 12 | grad_accum: 1
153


## 구성 — 데이터·모델

토크나이저+원본 로드 → `max_len` 절단(캐시) → 분류기 구성. 단계를 나눠 중간 점검·부분 재실행이 가능하다.

In [3]:
runner = TrainingRunner(cfg)

In [4]:
runner.load_data()        # 토크나이저 + 원본 데이터셋(prep 캐시 있으면 원본 생략)
runner.data.raw

[skip] prep 캐시 존재 — 원본 로드 생략: /workspace/prep_cache/axenc_len512


In [5]:
runner.prepare_data()     # max_len 절단 → prep 캐시
runner.data.dataset

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 201616
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11244
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11132
    })
})

In [6]:
runner.load_model()

[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertForSequenceClassification is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`
[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[model] skt/A.X-Encoder-base@9708f9c4(신규 헤드)


## OOM 확인

GPU/배치에서 안전한 `micro_batch` 상한을 실측

In [7]:
# probe_batches(
#     runner.model, runner.data.tokenizer.vocab_size, cfg.max_len,
#     train_mb=(32, 64, 96, 128, 160),
#     eval_mb=(128, 256, 512),
# )

## 훈련

In [8]:
runner.build_trainer()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


[schedule] 1576 step/epoch | eval·save 788 step마다(2회/epoch) | early stop 2 epoch(patience=4 eval)


In [9]:
runner.train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
788,0.001424,0.000942,0.604773,0.550824,0.511778,0.371362,0.666655
1576,0.000944,0.000771,0.688684,0.642999,0.626136,0.255570,0.726930
2364,0.000653,0.000532,0.779082,0.765240,0.763638,0.097467,0.761364
3152,0.000548,0.000496,0.794299,0.781741,0.791769,0.063690,0.768960
3940,0.000497,0.000508,0.791519,0.778037,0.777599,0.091628,0.768550
4728,0.000506,0.000475,0.801350,0.790909,0.798922,0.060636,0.774910
5516,0.000427,0.000473,0.804480,0.794255,0.794473,0.076267,0.784099
6304,0.000437,0.000443,0.811961,0.800614,0.809158,0.067194,0.791710
7092,0.000380,0.000449,0.819593,0.812118,0.817970,0.054258,0.787685
7880,0.000426,0.000455,0.816535,0.805349,0.809389,0.072584,0.794961


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 평가 · 메트릭 저장 · push

모델 가중치는 로컬에 두지 않고 Hub로만 올린다(팟을 지우면 로컬 사본은 사라진다). 로컬 사본이 필요하면 `runner.save_model()`.

In [10]:
test_metrics = runner.evaluate("test")   # 05_01 원본(0.8600) 대비 회귀 확인
for k, v in test_metrics.items():
    print(f"{k}: {v}")

[transformers] early stopping required metric_for_best_model, but did not find eval_micro_f1 so early stopping is disabled


Training Loss,Validation Loss,Step,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
0.000020,0.000871,18912,0.856996,0.854404,0.871395,0.012629,0.818365


test_loss: 0.0008710517431609333
test_micro_f1: 0.8569958847736625
test_macro_f1: 0.8544041801437436
test_sample_f1: 0.8713952195675781
test_empty_rate: 0.012628957666310921
test_anchor_weighted_f1: 0.8183645682006684


In [11]:
runner.save_metrics()     # runner.metrics(split 전체) → {tag}_metrics.json

[save] /workspace/output/modernbert-seed153/modernbert-patent-seed153_metrics.json  splits=['test']


In [12]:
runner.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

[push] ingyoun/A.X-patent-seed153


## val·test 로짓 덤프

`logits_{tag}_{split}.npy`를 `out_path` 상위(`/workspace/output/`)에 저장

In [13]:
runner.predict_logits("val")
runner.predict_logits("test")

[dump] /workspace/output/logits_modernbert-patent-seed153_val.npy  shape=(11132, 188)


[dump] /workspace/output/logits_modernbert-patent-seed153_test.npy  shape=(11244, 188)


array([[-1.84375, -5.09375, -6.34375, ..., -7.0625 , -6.8125 , -7.0625 ],
       [ 1.625  , -5.8125 , -5.6875 , ..., -6.75   , -6.875  , -7.28125],
       [ 4.4375 , -6.625  , -7.1875 , ..., -7.84375, -6.     , -7.34375],
       ...,
       [-8.0625 , -6.03125, -9.5625 , ..., -7.46875, -7.0625 ,  7.46875],
       [-8.625  , -7.28125, -9.4375 , ..., -8.8125 , -8.9375 ,  6.125  ],
       [-8.75   , -7.4375 , -9.875  , ..., -8.625  , -7.4375 ,  6.65625]],
      shape=(11244, 188), dtype=float32)